<a href="https://colab.research.google.com/github/hwanginseo04/-/blob/main/%EC%95%B1%ED%94%84%EB%A1%9C%EA%B7%B8%EB%9E%98%EB%B0%8D_%EC%A4%91%EA%B0%84%EA%B3%A0%EC%82%AC.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 필수 라이브러리 설치 (코랩 환경)
!pip install fastapi uvicorn httpx beautifulsoup4 gradio pandas deep-translator pyngrok nest_asyncio

import sqlite3
import httpx
import asyncio
import nest_asyncio
import uvicorn
import pandas as pd
import gradio as gr
from bs4 import BeautifulSoup
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from gradio import mount_gradio_app
from collections import Counter
from deep_translator import GoogleTranslator
from pyngrok import ngrok

# 코랩에서 비동기 서버를 돌리기 위한 설정
nest_asyncio.apply()

app = FastAPI(title="격언 관리 시스템 (중간고사)")
DB_NAME = "midterm_final.db"
translator = GoogleTranslator(source='en', target='ko')

# --- DB 및 테이블 초기화 ---
def init_db():
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    cursor.execute('''CREATE TABLE IF NOT EXISTS quotes
                      (id INTEGER PRIMARY KEY AUTOINCREMENT,
                       text TEXT, author TEXT, k_text TEXT)''')
    conn.commit()
    conn.close()

# --- API 기능 구현 (CRUD) ---
class QuoteUpdate(BaseModel):
    text: str
    author: str
    k_text: str

@app.post("/scrape", tags=["Management"])
async def scrape_quotes():
    count = 0
    conn = sqlite3.connect(DB_NAME)
    cursor = conn.cursor()
    async with httpx.AsyncClient(timeout=30.0) as client:
        for page in range(1, 3): # 2페이지까지 20개 수집
            url = f"https://quotes.toscrape.com/page/{page}/"
            response = await client.get(url)
            soup = BeautifulSoup(response.text, "html.parser")
            for q in soup.find_all("div", class_="quote"):
                if count >= 20: break
                text = q.find("span", class_="text").text
                author = q.find("small", class_="author").text
                try: k_text = translator.translate(text)
                except: k_text = text
                cursor.execute("INSERT INTO quotes (text, author, k_text) VALUES (?, ?, ?)", (text, author, k_text))
                count += 1
    conn.commit()
    conn.close()
    return {"message": f"{count}개 수집 및 번역 완료"}

@app.get("/quotes", tags=["CRUD"])
def get_quotes():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query("SELECT * FROM quotes", conn)
    conn.close()
    return df.to_dict(orient="records")

# --- Gradio UI 분석 기능 ---
def analyze_data():
    conn = sqlite3.connect(DB_NAME)
    df = pd.read_sql_query("SELECT * FROM quotes", conn)
    conn.close()
    if df.empty: return "데이터가 없습니다.", None, None

    words = " ".join(df['k_text'].tolist()).split()
    word_counts = Counter([w for w in words if len(w) > 1]).most_common(10)
    word_df = pd.DataFrame(word_counts, columns=['단어', '빈도'])

    author_counts = df['author'].value_counts().reset_index()
    author_counts.columns = ['저자', '명언 수']
    return "분석 완료!", word_df, author_counts

with gr.Blocks() as demo:
    gr.Markdown("# 🎓 중간고사 과제: 격언 분석 시스템")
    btn = gr.Button("데이터 분석 및 시각화 업데이트 🚀")
    status = gr.Textbox(label="상태")
    with gr.Row():
        plot1 = gr.BarPlot(x="단어", y="빈도", title="한국어 단어 빈도 TOP 10")
        plot2 = gr.BarPlot(x="저자", y="명언 수", title="저자별 분포 (추가 기능)")
    btn.click(analyze_data, outputs=[status, plot1, plot2])

app = mount_gradio_app(app, demo, path="/ui")

# --- 서버 실행 및 ngrok 배포 ---
if __name__ == "__main__":
    init_db()

    # ngrok 토큰 설정 (본인의 토큰을 입력하세요)
    NGROK_TOKEN = "3ChTmV0LuwFgxG0QlwxRqGbzvGZ_64cyaucWGFjK6uQBqbgGk" # 친구가 쓴 것과 동일하게 예시로 넣었습니다.
    ngrok.set_auth_token(NGROK_TOKEN)

    # 포트 8000 연결
    public_url = ngrok.connect(8000).public_url
    print(f"\n🚀 외부 접속 주소 (발표 시 사용):")
    print(f"📖 API 문서(Swagger): {public_url}/docs")
    print(f"🎨 사용자 UI(Gradio): {public_url}/ui\n")

    # FastAPI 서버 가동
    config = uvicorn.Config(app, host="0.0.0.0", port=8000)
    server = uvicorn.Server(config)
    loop = asyncio.get_event_loop()
    loop.create_task(server.serve())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 1.9 MB/s eta 0:00:00
new /ui

🚀 외부 접속 주소 (발표 시 사용):
📖 API 문서(Swagger): https://populace-drove-refurnish.ngrok-free.dev/docs
🎨 사용자 UI(Gradio): https://populace-drove-refurnish.ngrok-free.dev/ui

